In [1]:
import pandas as pd
import os
import glob
import re
from tqdm import tqdm
from fuzzywuzzy import fuzz
from datetime import datetime
import random

# Read in all files from AWS_Found folder - Link to credit data storage

In [2]:
found_path=r"C:\Users\F5448255\OneDrive - FRG\Documents\Web scrapping\aws_master\\"

In [3]:
path = found_path+"found" # use your path
all_files = glob.glob(os.path.join(path, "*.csv")) 
lis=[]
for file in tqdm(all_files):
    if file.endswith(".csv"):  
        df=pd.read_csv(file)
        lis.append(df)
df_1= pd.concat(lis, axis=0, ignore_index=True)
df_1.head(3)

  0%|          | 0/4332 [00:00<?, ?it/s]

100%|██████████| 4332/4332 [00:10<00:00, 412.81it/s]


,Our_Name,G_name,G_indus,G_ratings,G_reviews,ROW
0,PRYRA BELEGGINGS TRUST,NaN,NaN,Rated 4.0 out of 5,NaN,100019
1,P AND K GROUP PTY LTD,NaN,NaN,Rated 5.0 out of 5,NaN,100369
2,NEUROGENESIS PTY LTD,NaN,NaN,Rated 5.0 out of 5,NaN,100425


Combine with DDA table to get cutomer number

In [4]:
df_dda=pd.read_csv(found_path+"DDA_withrow_num.csv")
df_dda.rename(columns = {'row_num':'ROW'}, inplace = True)
df_dda.head(2)

,Unnamed: 0,CUST_NO,ACCT_NAME,SUM_of_YRLY_CR_TURNOVER,Found,ROW,CUST_NAME
0,201225,18423132,*ETC AGRO (PTY) LTD,8.534892e+09,NaN,33322,ETC AGRO PTY LTD
1,365342,23243872,*MINDSHARE SOUTH AFRICA,5.517779e+09,NaN,50554,MINDSHARE SOUTH AFRICA


Join on row_num

In [5]:
df=df_1.merge(df_dda,on="ROW",how="left")
df.head(3)

,Our_Name,G_name,G_indus,G_ratings,G_reviews,ROW,Unnamed: 0,CUST_NO,ACCT_NAME,SUM_of_YRLY_CR_TURNOVER,Found,CUST_NAME
0,PRYRA BELEGGINGS TRUST,NaN,NaN,Rated 4.0 out of 5,NaN,100019,658303,34315836,*JOJO TRADING ENTERPRISE,919217.31,NaN,JOJO TRADING ENTERPRISE
1,P AND K GROUP PTY LTD,NaN,NaN,Rated 5.0 out of 5,NaN,100369,660017,34397529,*MANIFEST SONS OF GOD TV,0.00,NaN,MANIFEST SONS OF GOD TV
2,NEUROGENESIS PTY LTD,NaN,NaN,Rated 5.0 out of 5,NaN,100425,660240,34408909,*K2013/175249 PTY LTD,315737.03,NaN,K2013175249 PTY LTD


In [6]:
drop_cols=['ROW', 'Unnamed: 0','ACCT_NAME','SUM_of_YRLY_CR_TURNOVER', 'Found', 'CUST_NAME']

In [7]:
df.drop(drop_cols, axis = 1, inplace = True)
df.head(3)

,Our_Name,G_name,G_indus,G_ratings,G_reviews,CUST_NO
0,PRYRA BELEGGINGS TRUST,NaN,NaN,Rated 4.0 out of 5,NaN,34315836
1,P AND K GROUP PTY LTD,NaN,NaN,Rated 5.0 out of 5,NaN,34397529
2,NEUROGENESIS PTY LTD,NaN,NaN,Rated 5.0 out of 5,NaN,34408909


In [8]:
#col_names=["CUST_NAME","CUST_NO","is_class","Google_name","Google_indus","Fuzzy_match","tried"]

Fuzzy Match score

In [9]:
df.rename(columns = {'Our_Name':'CUST_NAME',"CUST_NO":"CUST_NO","G_name":"Google_name","G_indus":"Google_indus"}, inplace = True)
df["Fuzzy_match"]=""
for row in tqdm(range(df.shape[0])):
    fnb_name=df.iloc[row,df.columns.get_loc("CUST_NAME")]
    google_name=df.iloc[row,df.columns.get_loc("Google_name")]
    
    df.iloc[row,df.columns.get_loc("Fuzzy_match")]=fuzz.token_sort_ratio(fnb_name,google_name)/100  
    
df["tried"]="Yes"
df["is_class"]="Yes"

df.head(2)

  8%|▊         | 347/4332 [00:00<00:01, 3443.93it/s]

100%|██████████| 4332/4332 [00:01<00:00, 4209.75it/s]


,CUST_NAME,Google_name,Google_indus,G_ratings,G_reviews,CUST_NO,Fuzzy_match,tried,is_class
0,PRYRA BELEGGINGS TRUST,NaN,NaN,Rated 4.0 out of 5,NaN,34315836,0.16,Yes,Yes
1,P AND K GROUP PTY LTD,NaN,NaN,Rated 5.0 out of 5,NaN,34397529,0.17,Yes,Yes


# Preprocessing

In [10]:
def preprocessing(sent):
    if isinstance(sent,str)==True and sent != "ABCXYZ":
        sent=sent.lower() #lower case everything
        sent = re.sub(r"\([^()]*\)", "", sent ) #remove brackets and words within
        sent=sent.replace("&", "and") #change & to and
        sent=re.sub(r'[^\w\s]', '', sent) #remove all punct
        sent=re.sub(r"\s+"," ", sent, flags = re.I) #remove mulit spaces
        
        
        list_word_remove=["nec","other","ltd","pty","cc"]
        wordlist=sent.split() ##Physiotherapy physiapy "other"
        sent=[x for x in wordlist if x not in list_word_remove]
        sent=' '.join(word for word in sent)
        sent=re.sub(r"\s+"," ", sent, flags = re.I) #remove mulit spaces    
    return sent

#administive services #$LOOOOOOOOOOOOK

def remove_in(string):
    '''
    Some Categories have locations built into them. In Johannesburg etc. If the word IN in presen,
    this code will remove in along will all the text that follows it. 
    
    ***Not included for now, data set cleans well with the in only
    As a last check, if a location is still present, it will be removed 
    '''
    
    list_string=string.split()
    try:
        index=list_string.index("in")
    except ValueError:
        index="None"
    
    if index != "None":
        str1 = " "
        string=(str1.join(list_string[0:index]))
            
    return string

def isolate_word(string,word): #isolate word
    list_string=string.split()
    try:
        index=list_string.index(word)#"restaurants"
    except ValueError:
        index="None"
    if index != "None":
        string=word
            
    return string

def remove_rest(string):
    list_string=string.split()
    try:
        index=list_string.index("restaurants")
    except ValueError:
        index="None"
    if index != "None":
        string="restaurants"
            
    return string

In [11]:
df = df[df['Google_indus'].notnull()]
df.shape

(257, 9)

In [12]:
#Some preprocessing
df["Category"]=df['Google_indus'].apply(lambda x:preprocessing(x))
df.head(3)

,CUST_NAME,Google_name,Google_indus,G_ratings,G_reviews,CUST_NO,Fuzzy_match,tried,is_class,Category
28,MERABE THE FIRM PTYLTD,Merabe The Firm,"Marketing agency in Soweto, South Africa",Rated 5.0 out of 5,1 Google review,35393457,0.81,Yes,Yes,marketing agency in soweto south africa
80,CLOCKWORK TRADING 1069,clockwork accounts,Accounting software company in Dublin,Rated 4.5 out of 5,2 Google reviews,8843930,0.5,Yes,Yes,accounting software company in dublin
120,CENTRAL LAKE TRADING 415,Central Lake Trading,"Leasing service in Krugersdorp, South Africa",NaN,NaN,38400516,0.91,Yes,Yes,leasing service in krugersdorp south africa


In [13]:
#Some preprocessing
df["Category"]=df['Google_indus'].apply(lambda x:preprocessing(x))
df["Category"]=df["Category"].apply(lambda x:remove_in(x))



#Remove the word in For Googel cards
isolate_words=["restaurants","cloths","clothing","hotel","restaurant"]
for word in isolate_words:
    df["Category"]=df["Category"].apply(lambda x:isolate_word(x,word))

df.head(2)


,CUST_NAME,Google_name,Google_indus,G_ratings,G_reviews,CUST_NO,Fuzzy_match,tried,is_class,Category
28,MERABE THE FIRM PTYLTD,Merabe The Firm,"Marketing agency in Soweto, South Africa",Rated 5.0 out of 5,1 Google review,35393457,0.81,Yes,Yes,marketing agency
80,CLOCKWORK TRADING 1069,clockwork accounts,Accounting software company in Dublin,Rated 4.5 out of 5,2 Google reviews,8843930,0.5,Yes,Yes,accounting software company


In [14]:
df["Category_s"]=df["Category"]
df["similarity"]=""
df.rename(columns = {'CUST_NAME':'left_side', 'Google_name':'right_side',"Fuzzy_match":"Final_Match"}, inplace = True)
df.head(2)

,left_side,right_side,Google_indus,G_ratings,G_reviews,CUST_NO,Final_Match,tried,is_class,Category,Category_s,similarity
28,MERABE THE FIRM PTYLTD,Merabe The Firm,"Marketing agency in Soweto, South Africa",Rated 5.0 out of 5,1 Google review,35393457,0.81,Yes,Yes,marketing agency,marketing agency,
80,CLOCKWORK TRADING 1069,clockwork accounts,Accounting software company in Dublin,Rated 4.5 out of 5,2 Google reviews,8843930,0.5,Yes,Yes,accounting software company,accounting software company,


# Save results - Change Date

In [15]:
save_string="AWS_Found_"+str(datetime.today().strftime('%Y_%m_%d'))+"_"+str(random.randint(1,20))+".csv"
save_string

'AWS_Found_2023_10_18_2.csv'

In [16]:
df.columns

Index(['left_side', 'right_side', 'Google_indus', 'G_ratings', 'G_reviews',
       'CUST_NO', 'Final_Match', 'tried', 'is_class', 'Category', 'Category_s',
       'similarity'],
      dtype='object')

In [17]:
df[["left_side","right_side","similarity",'G_ratings', 'G_reviews',
    "CUST_NO","Final_Match","Category_s",
    "Category"]].to_csv(found_path+save_string)

# Delete found files from the invoke folder

Still need to integrate with credit data storage

In [18]:
path_found =found_path+'found\\' # use your path

In [19]:
path_invoke=glob.glob(r"C:\Users\F5448255\OneDrive - FRG\Documents\Web scrapping\aws_master\dump\*/", recursive = True)
path_invoke

['C:\\Users\\F5448255\\OneDrive - FRG\\Documents\\Web scrapping\\aws_master\\dump\\001\\',
 'C:\\Users\\F5448255\\OneDrive - FRG\\Documents\\Web scrapping\\aws_master\\dump\\002\\',
 'C:\\Users\\F5448255\\OneDrive - FRG\\Documents\\Web scrapping\\aws_master\\dump\\003\\',
 'C:\\Users\\F5448255\\OneDrive - FRG\\Documents\\Web scrapping\\aws_master\\dump\\004\\',
 'C:\\Users\\F5448255\\OneDrive - FRG\\Documents\\Web scrapping\\aws_master\\dump\\005\\',
 'C:\\Users\\F5448255\\OneDrive - FRG\\Documents\\Web scrapping\\aws_master\\dump\\006\\',
 'C:\\Users\\F5448255\\OneDrive - FRG\\Documents\\Web scrapping\\aws_master\\dump\\007\\']

In [20]:
dict_location={}
for sub_folder in tqdm(path_invoke):
    all_files_invoke =  glob.glob(os.path.join(sub_folder, "*.csv")) 
    for file in all_files_invoke:
        dict_location[file[len(sub_folder):]]=sub_folder

100%|██████████| 7/7 [00:04<00:00,  1.69it/s]


In [21]:
all_files = glob.glob(os.path.join(path_found, "*.csv")) 

lis=[]
count_=0
cp=0
for file in tqdm(all_files):
    if file.endswith(".csv"):  
        #Search found files
        df=pd.read_csv(file)
        name=df.iloc[0,df.columns.get_loc("Our_Name")]
        row_num=df.iloc[0,df.columns.get_loc("ROW")]
        file_name=str(row_num)+".csv"

        if dict_location.get(file_name,0) !=0:
            try:
                count_+=1
                os.remove(dict_location.get(file_name,0)+file_name)
                if cp<5:
                    print(dict_location.get(file_name,0)+file_name)
                    cp+=1

            except OSError as e:  ## if failed, report it back to the user ##
                print ("Error: %s - %s." % (e.filename, e.strerror))     
print("Files removed :" , count_)

  0%|          | 14/4332 [00:00<00:31, 137.54it/s]

C:\Users\F5448255\OneDrive - FRG\Documents\Web scrapping\aws_master\dump\001\100019.csv
C:\Users\F5448255\OneDrive - FRG\Documents\Web scrapping\aws_master\dump\001\100369.csv
C:\Users\F5448255\OneDrive - FRG\Documents\Web scrapping\aws_master\dump\001\100425.csv
C:\Users\F5448255\OneDrive - FRG\Documents\Web scrapping\aws_master\dump\001\100481.csv
C:\Users\F5448255\OneDrive - FRG\Documents\Web scrapping\aws_master\dump\001\100504.csv


100%|██████████| 4332/4332 [00:28<00:00, 154.30it/s]

Files removed : 4332


Remove files from path_found

In [22]:
import shutil
shutil.rmtree(path_found)  
os.mkdir(path_found)

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\F5448255\\OneDrive - FRG\\Documents\\Web scrapping\\aws_master\\\\found\\'